# 05 — Virtual Max synthetic scenario

AIDP operational pipeline notebook. It uses Volumes and governed tables only; the Git checkout is code only.

In [ ]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')
sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import LANDING_COUNTS, RAW_SESSIONS, STRUCTURAL_KEYS, landing_paths, load_aidp_config, missing_landing_paths, table_name
config = load_aidp_config(root / 'config/settings.yaml')


In [ ]:
profiles = spark.table(table_name(config, 'gold_historical_schema', 'observed_driver_profiles')).filter("support_status = 'sufficient'").orderBy('pooled_median_pace').limit(2).collect()
if len(profiles) != 2: raise ValueError('Historical profiles are unavailable or insufficient')
fastest, second = profiles
advantage = float(second.pooled_median_pace - fastest.pooled_median_pace)
virtual_pace = float(fastest.pooled_median_pace - advantage)
scenario = spark.createDataFrame([('VIRTUAL_MAX', int(fastest.driver_id), float(fastest.pooled_median_pace), advantage, virtual_pace, 'SYNTHETIC', 'WHAT-IF', 'NOT HISTORICAL')], ['scenario_driver_id','reference_driver_id','reference_pace_seconds','assumed_advantage_seconds_per_lap','virtual_max_pace_seconds','scenario_type','label','historical_status'])
scenario.write.mode('overwrite').saveAsTable(table_name(config, 'gold_scenario_schema', 'virtual_max_profile'))
scenario.show(truncate=False)
print('Regression targets only: P1/P2 ≈ 0.2205; Virtual Max pace ≈ 80.0865. No real Verstappen performance is claimed.')
